# Aula 05 — XML: Extensible Markup Language
**QXD0099 - Desenvolvimento de Software para Persistência**  
Universidade Federal do Ceará - Campus Quixadá  
Prof. Francisco Victor da Silva Pinheiro  
victorpinheiro@ufc.br

> Notebook prático e comentado para execução em sala no Google Colab.

## Agenda
- Motivação
- O que é XML
- Modelo de dados hierárquico
- Elementos
- Atributos
- Estrutura de um documento XML
- Manipulações
- APIs para processamento de XML
- ElementTree
- Minidom
- lxml
- BeautifulSoup
- xmltodict
- CRUD persistindo em XML

# 1. Motivação

XML surgiu para facilitar a **representação, troca e interpretação automática de dados** entre sistemas.

Enquanto HTML é voltado principalmente para apresentação, XML permite descrever o significado dos dados.

```xml
<livro>
    <titulo>Inside XML</titulo>
    <autor>Steven Holzner</autor>
    <preco>150.00</preco>
</livro>
```

# 2. O que é XML?

XML significa **eXtensible Markup Language**.

Características:

- linguagem de marcação;
- padrão recomendado pelo W3C;
- representação e troca de dados;
- extensível;
- hierárquico;
- legível por humanos e máquinas.

## 3. Criando um primeiro documento XML

In [1]:
xml_livro = """<?xml version="1.0" encoding="UTF-8"?>
<livros>
    <livro>
        <ISBN>1234</ISBN>
        <titulo>Inside XML</titulo>
        <editora>New Riders</editora>
        <edicao>3</edicao>

        <autor>
            <nome>Steven</nome>
            <endereco>NY</endereco>
        </autor>

        <autor>
            <nome>Holzner</nome>
            <endereco>Miami</endereco>
        </autor>
    </livro>
</livros>
"""

with open("livros.xml", "w", encoding="utf-8") as arquivo:
    arquivo.write(xml_livro)

print("Arquivo livros.xml criado.")

Arquivo livros.xml criado.


## 4. Visualizando o XML

In [2]:
with open("livros.xml", "r", encoding="utf-8") as arquivo:
    print(arquivo.read())

<?xml version="1.0" encoding="UTF-8"?>
<livros>
    <livro>
        <ISBN>1234</ISBN>
        <titulo>Inside XML</titulo>
        <editora>New Riders</editora>
        <edicao>3</edicao>

        <autor>
            <nome>Steven</nome>
            <endereco>NY</endereco>
        </autor>

        <autor>
            <nome>Holzner</nome>
            <endereco>Miami</endereco>
        </autor>
    </livro>
</livros>



# 5. Modelo hierárquico

```text
livros
└── livro
    ├── ISBN
    ├── titulo
    ├── editora
    ├── edicao
    ├── autor
    │   ├── nome
    │   └── endereco
    └── autor
        ├── nome
        └── endereco
```

XML é frequentemente entendido como uma árvore.

## 6. Elementos

```xml
<titulo>Inside XML</titulo>
```

- tag inicial: `<titulo>`;
- conteúdo: `Inside XML`;
- tag final: `</titulo>`.

## 7. Elementos vazios

```xml
<fone/>
```

ou:

```xml
<fone></fone>
```

# 8. Atributos

```xml
<livro isbn="85.241.0591-9">
    <titulo>Inside XML</titulo>
</livro>
```

Um atributo é um par **nome = valor**.

In [3]:
xml_atributos = """<?xml version="1.0" encoding="UTF-8"?>
<livros>
    <livro isbn="85.241.0591-9" idioma="pt-BR">
        <titulo>Inside XML</titulo>
        <autor>Steven Holzner</autor>
        <preco>150.00</preco>
    </livro>

    <livro isbn="978-1234567890" idioma="en">
        <titulo>XML Fundamentals</titulo>
        <autor>Maria Silva</autor>
        <preco>120.00</preco>
    </livro>
</livros>
"""

with open("livros_atributos.xml", "w", encoding="utf-8") as arquivo:
    arquivo.write(xml_atributos)

print("livros_atributos.xml criado.")

livros_atributos.xml criado.


# 9. Regras de um XML bem-formado

- um único elemento raiz;
- todas as tags devem ser fechadas;
- tags corretamente aninhadas;
- XML é case sensitive;
- atributos entre aspas;
- cada elemento, exceto a raiz, possui um pai.

## 10. XML incorreto

In [4]:
xml_incorreto = """
<cliente>
    <nome>Ana
    <email>ana@email.com</email>
</cliente>
"""

print(xml_incorreto)


<cliente>
    <nome>Ana
    <email>ana@email.com</email>
</cliente>



## 11. Detectando erro de formação

In [5]:
import xml.etree.ElementTree as ET

try:
    ET.fromstring(xml_incorreto)
    print("XML válido.")

except ET.ParseError as erro:
    print("XML malformado.")
    print("Erro:", erro)

XML malformado.
Erro: mismatched tag: line 5, column 2


# 12. XML é case sensitive

`<Nome>` e `<nome>` são tags diferentes.

# 13. ElementTree

`xml.etree.ElementTree` faz parte da biblioteca padrão do Python.

Permite ler, navegar, criar, alterar e salvar XML.

In [6]:
import xml.etree.ElementTree as ET

tree = ET.parse("livros_atributos.xml")
root = tree.getroot()

print("Tag raiz:", root.tag)
print("Atributos da raiz:", root.attrib)

Tag raiz: livros
Atributos da raiz: {}


## 14. Percorrendo filhos da raiz

In [7]:
for elemento in root:
    print("Tag:", elemento.tag)
    print("Atributos:", elemento.attrib)
    print("---")

Tag: livro
Atributos: {'isbn': '85.241.0591-9', 'idioma': 'pt-BR'}
---
Tag: livro
Atributos: {'isbn': '978-1234567890', 'idioma': 'en'}
---


## 15. Acessando atributos

In [8]:
for livro in root.findall("livro"):
    print("ISBN:", livro.get("isbn"))
    print("Idioma:", livro.get("idioma"))
    print("---")

ISBN: 85.241.0591-9
Idioma: pt-BR
---
ISBN: 978-1234567890
Idioma: en
---


## 16. Acessando filhos

In [9]:
for livro in root.findall("livro"):

    titulo = livro.find("titulo").text
    autor = livro.find("autor").text
    preco = livro.find("preco").text

    print("Título:", titulo)
    print("Autor:", autor)
    print("Preço:", preco)
    print("---")

Título: Inside XML
Autor: Steven Holzner
Preço: 150.00
---
Título: XML Fundamentals
Autor: Maria Silva
Preço: 120.00
---


## 17. Busca em qualquer nível

In [10]:
titulos = root.findall(".//titulo")

for titulo in titulos:
    print(titulo.text)

Inside XML
XML Fundamentals


## 18. Conversão de tipos

In [11]:
for livro in root.findall("livro"):

    titulo = livro.find("titulo").text

    # XML entrega conteúdo textual como string.
    preco = float(livro.find("preco").text)

    print(titulo, "-", preco, type(preco))

Inside XML - 150.0 <class 'float'>
XML Fundamentals - 120.0 <class 'float'>


# 19. Criando XML programaticamente

In [12]:
import xml.etree.ElementTree as ET

root_novo = ET.Element("alunos")

aluno1 = ET.SubElement(root_novo, "aluno", {"id": "1"})
ET.SubElement(aluno1, "nome").text = "Ana"
ET.SubElement(aluno1, "curso").text = "Engenharia de Software"
ET.SubElement(aluno1, "nota").text = "8.5"

aluno2 = ET.SubElement(root_novo, "aluno", {"id": "2"})
ET.SubElement(aluno2, "nome").text = "Bruno"
ET.SubElement(aluno2, "curso").text = "Sistemas de Informação"
ET.SubElement(aluno2, "nota").text = "7.0"

tree_novo = ET.ElementTree(root_novo)

tree_novo.write(
    "alunos.xml",
    encoding="utf-8",
    xml_declaration=True
)

print("alunos.xml criado.")

alunos.xml criado.


## 20. Formatando o XML

In [13]:
tree = ET.parse("alunos.xml")

ET.indent(tree, space="    ")

tree.write(
    "alunos_formatado.xml",
    encoding="utf-8",
    xml_declaration=True
)

with open("alunos_formatado.xml", "r", encoding="utf-8") as arquivo:
    print(arquivo.read())

<?xml version='1.0' encoding='utf-8'?>
<alunos>
    <aluno id="1">
        <nome>Ana</nome>
        <curso>Engenharia de Software</curso>
        <nota>8.5</nota>
    </aluno>
    <aluno id="2">
        <nome>Bruno</nome>
        <curso>Sistemas de Informação</curso>
        <nota>7.0</nota>
    </aluno>
</alunos>


# 21. Minidom

`xml.dom.minidom` implementa uma interface DOM e é útil para navegação e formatação.

In [14]:
from xml.dom import minidom

doc = minidom.parse("alunos.xml")
root_dom = doc.documentElement

print("Elemento raiz:", root_dom.tagName)

Elemento raiz: alunos


## 22. Obtendo elementos com Minidom

In [15]:
alunos = doc.getElementsByTagName("aluno")

for aluno in alunos:

    print("ID:", aluno.getAttribute("id"))

    nomes = aluno.getElementsByTagName("nome")

    if nomes:
        print("Nome:", nomes[0].firstChild.nodeValue)

    print("---")

ID: 1
Nome: Ana
---
ID: 2
Nome: Bruno
---


## 23. XML formatado com Minidom

In [16]:
xml_formatado = doc.toprettyxml(indent="    ")

print(xml_formatado)

<?xml version="1.0" ?>
<alunos>
    <aluno id="1">
        <nome>Ana</nome>
        <curso>Engenharia de Software</curso>
        <nota>8.5</nota>
    </aluno>
    <aluno id="2">
        <nome>Bruno</nome>
        <curso>Sistemas de Informação</curso>
        <nota>7.0</nota>
    </aluno>
</alunos>



# 24. lxml

`lxml` é uma biblioteca externa eficiente e poderosa.

Suporta, entre outros recursos, XPath e XSLT.

In [17]:
!pip -q install lxml

In [18]:
from lxml import etree

tree_lxml = etree.parse("livros_atributos.xml")
root_lxml = tree_lxml.getroot()

for element in root_lxml:
    print("Tag:", element.tag, "| Atributos:", element.attrib)

Tag: livro | Atributos: {'isbn': '85.241.0591-9', 'idioma': 'pt-BR'}
Tag: livro | Atributos: {'isbn': '978-1234567890', 'idioma': 'en'}


## 25. XPath

In [19]:
titulos = tree_lxml.xpath(
    "//livro/titulo/text()"
)

print(titulos)

['Inside XML', 'XML Fundamentals']


## 26. XPath com filtro

In [20]:
resultado = tree_lxml.xpath(
    "//livro[@idioma='pt-BR']/titulo/text()"
)

print(resultado)

['Inside XML']


# 27. BeautifulSoup para XML

Oferece uma interface de busca bastante simples.

In [21]:
!pip -q install beautifulsoup4

In [22]:
from bs4 import BeautifulSoup

with open("livros_atributos.xml", "r", encoding="utf-8") as arquivo:
    soup = BeautifulSoup(arquivo, "xml")

for livro in soup.find_all("livro"):

    print("ISBN:", livro.get("isbn"))
    print("Título:", livro.find("titulo").text)
    print("Autor:", livro.find("autor").text)
    print("---")

ISBN: 85.241.0591-9
Título: Inside XML
Autor: Steven Holzner
---
ISBN: 978-1234567890
Título: XML Fundamentals
Autor: Maria Silva
---


# 28. xmltodict

Converte XML para estruturas Python semelhantes a dicionários.

In [23]:
!pip -q install xmltodict

In [24]:
import xmltodict

with open("livros_atributos.xml", "r", encoding="utf-8") as arquivo:
    data = xmltodict.parse(arquivo.read())

print(type(data))
print(data)

<class 'dict'>
{'livros': {'livro': [{'@isbn': '85.241.0591-9', '@idioma': 'pt-BR', 'titulo': 'Inside XML', 'autor': 'Steven Holzner', 'preco': '150.00'}, {'@isbn': '978-1234567890', '@idioma': 'en', 'titulo': 'XML Fundamentals', 'autor': 'Maria Silva', 'preco': '120.00'}]}}


## 29. Navegando pelo resultado

In [25]:
livros = data["livros"]["livro"]

for livro in livros:

    print("ISBN:", livro["@isbn"])
    print("Idioma:", livro["@idioma"])
    print("Título:", livro["titulo"])
    print("Autor:", livro["autor"])
    print("---")

ISBN: 85.241.0591-9
Idioma: pt-BR
Título: Inside XML
Autor: Steven Holzner
---
ISBN: 978-1234567890
Idioma: en
Título: XML Fundamentals
Autor: Maria Silva
---


# 30. CRUD persistindo em XML

CRUD:

- **Create**
- **Read**
- **Update**
- **Delete**

Vamos persistir produtos em `produtos.xml`.

## 31. Modelo Produto

In [26]:
from dataclasses import dataclass

@dataclass
class Produto:
    id: int
    nome: str
    preco: float
    quantidade: int

produto = Produto(
    id=1,
    nome="Teclado",
    preco=250.0,
    quantidade=10
)

print(produto)

Produto(id=1, nome='Teclado', preco=250.0, quantidade=10)


## 32. Arquivo de persistência

In [27]:
XML_FILE = "produtos.xml"

print(XML_FILE)

produtos.xml


## 33. Escrita no XML

In [28]:
import xml.etree.ElementTree as ET

def escrever_dados_xml(produtos):

    root = ET.Element("produtos")

    for produto in produtos:

        produto_elem = ET.SubElement(root, "produto")

        ET.SubElement(
            produto_elem,
            "id"
        ).text = str(produto.id)

        ET.SubElement(
            produto_elem,
            "nome"
        ).text = produto.nome

        ET.SubElement(
            produto_elem,
            "preco"
        ).text = str(produto.preco)

        ET.SubElement(
            produto_elem,
            "quantidade"
        ).text = str(produto.quantidade)

    tree = ET.ElementTree(root)

    ET.indent(tree, space="    ")

    tree.write(
        XML_FILE,
        encoding="utf-8",
        xml_declaration=True
    )

## 34. Leitura do XML

In [29]:
import os
import xml.etree.ElementTree as ET

def ler_dados_xml():

    produtos = []

    if not os.path.exists(XML_FILE):
        return produtos

    tree = ET.parse(XML_FILE)
    root = tree.getroot()

    for elem in root.findall("produto"):

        produto = Produto(
            id=int(elem.find("id").text),
            nome=elem.find("nome").text,
            preco=float(elem.find("preco").text),
            quantidade=int(elem.find("quantidade").text)
        )

        produtos.append(produto)

    return produtos

## 35. Dados iniciais

In [30]:
produtos = [
    Produto(1, "Teclado", 250.0, 10),
    Produto(2, "Mouse", 80.0, 20),
    Produto(3, "Monitor", 1200.0, 5)
]

escrever_dados_xml(produtos)

print("Produtos persistidos.")

Produtos persistidos.


## 36. Visualizando `produtos.xml`

In [31]:
with open(XML_FILE, "r", encoding="utf-8") as arquivo:
    print(arquivo.read())

<?xml version='1.0' encoding='utf-8'?>
<produtos>
    <produto>
        <id>1</id>
        <nome>Teclado</nome>
        <preco>250.0</preco>
        <quantidade>10</quantidade>
    </produto>
    <produto>
        <id>2</id>
        <nome>Mouse</nome>
        <preco>80.0</preco>
        <quantidade>20</quantidade>
    </produto>
    <produto>
        <id>3</id>
        <nome>Monitor</nome>
        <preco>1200.0</preco>
        <quantidade>5</quantidade>
    </produto>
</produtos>


## 37. READ — listar produtos

In [32]:
for produto in ler_dados_xml():
    print(produto)

Produto(id=1, nome='Teclado', preco=250.0, quantidade=10)
Produto(id=2, nome='Mouse', preco=80.0, quantidade=20)
Produto(id=3, nome='Monitor', preco=1200.0, quantidade=5)


## 38. CREATE — adicionar produto

In [33]:
def adicionar_produto(produto):

    produtos = ler_dados_xml()

    for existente in produtos:
        if existente.id == produto.id:
            return False

    produtos.append(produto)
    escrever_dados_xml(produtos)

    return True

In [34]:
novo_produto = Produto(
    4,
    "Webcam",
    320.0,
    8
)

print(
    "Produto adicionado."
    if adicionar_produto(novo_produto)
    else "ID já existente."
)

Produto adicionado.


## 39. READ — buscar por ID

In [35]:
def buscar_produto(id_produto):

    for produto in ler_dados_xml():

        if produto.id == id_produto:
            return produto

    return None

print(buscar_produto(2))

Produto(id=2, nome='Mouse', preco=80.0, quantidade=20)


## 40. UPDATE — atualizar produto

In [36]:
def atualizar_produto(
    id_produto,
    novo_nome=None,
    novo_preco=None,
    nova_quantidade=None
):

    produtos = ler_dados_xml()

    for produto in produtos:

        if produto.id == id_produto:

            if novo_nome is not None:
                produto.nome = novo_nome

            if novo_preco is not None:
                produto.preco = novo_preco

            if nova_quantidade is not None:
                produto.quantidade = nova_quantidade

            escrever_dados_xml(produtos)
            return True

    return False

In [37]:
atualizar_produto(
    2,
    novo_preco=95.0,
    nova_quantidade=30
)

print(buscar_produto(2))

Produto(id=2, nome='Mouse', preco=95.0, quantidade=30)


## 41. DELETE — remover produto

In [38]:
def remover_produto(id_produto):

    produtos = ler_dados_xml()

    quantidade_antes = len(produtos)

    produtos = [
        produto
        for produto in produtos
        if produto.id != id_produto
    ]

    if len(produtos) == quantidade_antes:
        return False

    escrever_dados_xml(produtos)

    return True

In [39]:
print(
    "Produto removido."
    if remover_produto(1)
    else "Produto não encontrado."
)

for produto in ler_dados_xml():
    print(produto)

Produto removido.
Produto(id=2, nome='Mouse', preco=95.0, quantidade=30)
Produto(id=3, nome='Monitor', preco=1200.0, quantidade=5)
Produto(id=4, nome='Webcam', preco=320.0, quantidade=8)


# 42. CRUD completo com menu

In [40]:
def listar_produtos():

    produtos = ler_dados_xml()

    if not produtos:
        print("Nenhum produto cadastrado.")
        return

    print("\n=== PRODUTOS ===")

    for produto in produtos:
        print(
            f"ID: {produto.id} | "
            f"Nome: {produto.nome} | "
            f"Preço: R$ {produto.preco:.2f} | "
            f"Quantidade: {produto.quantidade}"
        )


def cadastrar_produto():

    try:
        id_produto = int(input("ID: "))
        nome = input("Nome: ").strip()
        preco = float(input("Preço: "))
        quantidade = int(input("Quantidade: "))

    except ValueError:
        print("Dados inválidos.")
        return

    produto = Produto(
        id_produto,
        nome,
        preco,
        quantidade
    )

    if adicionar_produto(produto):
        print("Produto cadastrado.")
    else:
        print("ID já existente.")


def consultar_produto():

    try:
        id_produto = int(input("ID: "))

    except ValueError:
        print("ID inválido.")
        return

    produto = buscar_produto(id_produto)

    if produto:
        print(produto)
    else:
        print("Produto não encontrado.")


def editar_produto():

    try:
        id_produto = int(input("ID: "))

    except ValueError:
        print("ID inválido.")
        return

    produto = buscar_produto(id_produto)

    if not produto:
        print("Produto não encontrado.")
        return

    print("Deixe vazio para manter o valor atual.")

    nome = input(
        f"Nome [{produto.nome}]: "
    ).strip()

    preco_texto = input(
        f"Preço [{produto.preco}]: "
    ).strip()

    qtd_texto = input(
        f"Quantidade [{produto.quantidade}]: "
    ).strip()

    try:
        novo_preco = float(preco_texto) if preco_texto else None
        nova_quantidade = int(qtd_texto) if qtd_texto else None

    except ValueError:
        print("Valor inválido.")
        return

    atualizar_produto(
        id_produto,
        novo_nome=nome if nome else None,
        novo_preco=novo_preco,
        nova_quantidade=nova_quantidade
    )

    print("Produto atualizado.")


def excluir_produto():

    try:
        id_produto = int(input("ID: "))

    except ValueError:
        print("ID inválido.")
        return

    if remover_produto(id_produto):
        print("Produto removido.")
    else:
        print("Produto não encontrado.")

## 43. Executando o menu

In [ ]:
while True:

    print("\n==========================")
    print("   CRUD DE PRODUTOS XML")
    print("==========================")
    print("1 - Cadastrar")
    print("2 - Listar")
    print("3 - Buscar")
    print("4 - Atualizar")
    print("5 - Remover")
    print("0 - Sair")

    opcao = input("Escolha uma opção: ")

    if opcao == "1":
        cadastrar_produto()

    elif opcao == "2":
        listar_produtos()

    elif opcao == "3":
        consultar_produto()

    elif opcao == "4":
        editar_produto()

    elif opcao == "5":
        excluir_produto()

    elif opcao == "0":
        print("Programa encerrado.")
        break

    else:
        print("Opção inválida.")


   CRUD DE PRODUTOS XML
1 - Cadastrar
2 - Listar
3 - Buscar
4 - Atualizar
5 - Remover
0 - Sair
Escolha uma opção: 1
ID: 0
Nome: 0


# 44. Atividade prática

Evolua o CRUD adicionando ao produto:

- categoria;
- fabricante.

XML esperado:

```xml
<produto>
    <id>1</id>
    <nome>Teclado</nome>
    <categoria>Periféricos</categoria>
    <fabricante>Logitech</fabricante>
    <preco>250.0</preco>
    <quantidade>10</quantidade>
</produto>
```

Adapte leitura, escrita, cadastro, consulta e atualização.

# Fechamento

```text
XML
 │
 ├── Elementos
 ├── Atributos
 ├── Hierarquia
 │
 ├── ElementTree
 ├── Minidom
 ├── lxml
 ├── BeautifulSoup
 ├── xmltodict
 │
 └── CRUD persistindo em XML
```

## Pontos principais

- XML é extensível;
- possui estrutura hierárquica;
- todo documento deve ter raiz;
- tags devem estar bem-formadas;
- XML é case sensitive;
- atributos adicionam metadados;
- ElementTree cobre muitos casos cotidianos;
- Minidom implementa DOM;
- lxml acrescenta recursos como XPath;
- BeautifulSoup simplifica buscas;
- xmltodict aproxima XML de dicionários Python;
- XML pode funcionar como um mecanismo simples de persistência.